# 02 · Anomaly detectors, side by side

There is no labelled ground truth, so this notebook only shows *what* each detector flags and how much they agree.

In [ ]:
import sys
sys.path.insert(0, "../src")
import pandas as pd
import matplotlib.pyplot as plt
from incubator.load import load_history, load_stress_log, summary
from incubator import features as fe, anomaly as an, thermal_fit as tf
pd.set_option("display.width", 160)

In [ ]:
df = load_history()
ft = fe.feature_table(df)
ft.describe().T

## Detector 1 · rolling z-score of the temperature slope, per heater state

In [ ]:
z = an.rolling_zscore(ft, window=40, threshold=3.0)
an.flagged_episodes(z['flag'])

## Detector 2 · IsolationForest on the feature table

In [ ]:
iso = an.isolation_forest(ft, contamination=0.03)
an.flagged_episodes(iso['flag'])

## Agreement

In [ ]:
an.compare(z['flag'], iso['flag']).as_dict()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(df.index, df['temp'], lw=1)
zf = ft.index[z['flag']]; isf = ft.index[iso['flag']]
ax.scatter(zf, df.loc[zf, 'temp'], facecolors='none', edgecolors='C3', label='z-score')
ax.scatter(isf, df.loc[isf, 'temp'] - 0.3, marker='v', color='C1', label='IsolationForest')
ax.legend();

## What the flagged readings look like
Try changing `window`, `threshold` and `contamination` above and re-running; the sets change a lot, which is the point: unsupervised 'anomalies' are a choice of definition, not a fact about the data.

In [ ]:
ft.loc[z['flag']].head(20)

In [ ]:
ft.loc[iso['flag']].head(20)

## Thermal model fit (used by the control-simulation repo)

In [ ]:
fit = tf.fit_first_order(df)
fit.as_dict()